## Seq2Seq Encoder - Decoder Model with Attention

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import TensorDataset, DataLoader
import lightning as L

## Creating the Dataset

In [2]:
# We create a dictionary that maps vocabulary tokens to id numbers
english_token_to_id = {'lets':0,
                       'to': 1,
                       'go': 2,
                       '<EOS': 3
}

# Next we create a dictionary that maps the ids to tokens
english_id_to_token = dict(map(reversed, english_token_to_id.items()))

spanish_token_to_id = {'ir': 0,
                       'vamos': 1,
                       'y': 2,
                       '<EOS>': 3}

spanish_id_to_token = dict(map(reversed, spanish_token_to_id.items()))

inputs = torch.tensor([[english_token_to_id["lets"],
                        english_token_to_id["go"]],
                        
                        [english_token_to_id["to"],
                        english_token_to_id["go"]]])

labels = torch.tensor([[spanish_token_to_id["vamos"],
                        spanish_token_to_id["<EOS>"]],
                        
                        [spanish_token_to_id["ir"],
                        spanish_token_to_id["<EOS>"]]])


dataset = TensorDataset(inputs, labels)
dataloader = DataLoader(dataset)

In [3]:
class seq2seq_attention(L.LightningModule):
    def __init__(self, max_len=2):
        super().__init__()
        self.max_decoder_length = max_len
        L.seed_everything(seed=42)

        # Encoder
        self.encoder_we = nn.Embedding(num_embeddings=4,
                                      embedding_dim=1)
        
        self.encoder_lstm = nn.LSTM(input_size=1,
                                      hidden_size=1,
                                      num_layers=1)
        
        # Decoding
        self.decoder_we = nn.Embedding(num_embeddings=4,
                                      embedding_dim=1)
        self.decoder_lstm = nn.LSTM(input_size=1,
                                      hidden_size=1,
                                      num_layers=1)
        
        self.decoder_fc = nn.Linear(in_features=2,
                                   out_features=4)
        
        self.loss = nn.CrossEntropyLoss()

    def forward(self, input, output=None):
        # Encoding
        encoder_embeddings = self.encoder_we(input)
        encoder_lstm_output, (encoder_lstm_hidden, encoder_lstm_cell) = self.encoder_lstm(encoder_embeddings)

        # Decoding
        decoder_token_id = torch.tensor([spanish_token_to_id["<EOS>"]])
        decoder_embeddings = self.decoder_we(decoder_token_id)
        decoder_lstm_output, (decoder_lstm_hidden, decoder_lstm_cell) = self.decoder_lstm(decoder_embeddings,
                                                                                         (encoder_lstm_hidden,
                                                                                         encoder_lstm_cell))
        # Calculate the attention using dot product (unnormalized cosine similarity)
        # We try to find words in the input that are used in the same context as those proposed by the lstm 
        sims = torch.matmul(decoder_lstm_output, encoder_lstm_output.transpose(dim0=0, dim1=1))
        
        # Then we apply Softmax to determine what percent of each token's value to use in the final attention values
        attention_percents = F.softmax(sims, dim=1)

        # Next we scale the values by theri assiciated percentages and add them up
        attention_values = torch.matmul(attention_percents, encoder_lstm_output) # encoder_lstm_output is the short term memory at every timestep

        # Finally we concatenate the attention values with the short term memories from the first decoder lstm
        values_to_fc_layer = torch.cat((attention_values, decoder_lstm_output), 1)

        output_values = self.decoder_fc(values_to_fc_layer)
        outputs = output_values
        predicted_id = torch.tensor([torch.argmax(output_values)])
        predicted_ids = predicted_id

        for i in range(1,self.max_decoder_length):
            if (output == None):
                if (predicted_id == spanish_token_to_id["<EOS>"]):
                    break
                decoder_embeddings = self.decoder_we(predicted_id)
            else:
                decoder_embeddings = self.decoder_we(torch.tensor([output[i-1]])) 

            decoder_lstm_output, (decoder_lstm_hidden, decoder_lstm_cell) = self.decoder_lstm(decoder_embeddings,
                                                                                              (decoder_lstm_hidden,
                                                                                              decoder_lstm_cell))
            sims = torch.matmul(decoder_lstm_output, encoder_lstm_output.transpose(dim0=0, dim1=1))
            attention_percents = F.softmax(sims, dim=1)
            attention_values = torch.matmul(attention_percents, encoder_lstm_output)
            values_to_fc_layer = torch.cat((attention_values, decoder_lstm_output), 1)

            output_values = self.decoder_fc(values_to_fc_layer)
            outputs = torch.cat((outputs, output_values), 0)
            predicted_id = torch.tensor([torch.argmax(output_values)])
            predicted_ids = torch.cat((predicted_ids, predicted_id))

        return (outputs)
    
    def configure_optimizers(self):
        return Adam(self.parameters(), lr=0.1)
    
    def training_step(self, batch, batch_idx):
        input_tokens, labes = batch
        output = self.forward(input_tokens[0], labels[0])
        loss = self.loss(output, labels[0])
        return loss
            
                



In [4]:
model = seq2seq_attention()
trainer = L.Trainer(max_epochs=20, accelerator="cpu")
trainer.fit(model, train_dataloaders=dataloader)

Seed set to 42
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/Users/odysseasgeorgiades/Desktop/Neural Networks/.venv/lib/python3.14/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.

  | Name         | Type             | Params | Mode  | FLOPs
------------------------------------------------------------------
0 | encoder_we   | Embedding        | 4      | train | 0    
1 | encoder_lstm | LSTM             | 16     | train | 0    
2 | decoder_

Epoch 19: 100%|██████████| 2/2 [00:00<00:00, 557.38it/s, v_num=1]

`Trainer.fit` stopped: `max_epochs=20` reached.


Epoch 19: 100%|██████████| 2/2 [00:00<00:00, 347.34it/s, v_num=1]


In [5]:

inputs = torch.tensor([
    english_token_to_id['lets'],
    english_token_to_id['go']
])

outputs = model.forward(input=inputs, output=None)

print("Translated text...")
predicted_ids = torch.argmax(outputs, dim=1)
for id in predicted_ids:
    print("\t", spanish_id_to_token[id.item()])

Translated text...
	 vamos
	 <EOS>


## Saving the trained model weights and biases

In [6]:
# Count the number of parameters we had to train 
total_trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total number of trainable parameters: ", total_trainable_params)

Total number of trainable parameters:  52


In [7]:
# Printing out Weight and Biases
print("Printing out Weight and Biases...")
for name, param in model.named_parameters():
    print(name, param.data)


Printing out Weight and Biases...
encoder_we.weight tensor([[1.3082],
        [1.2784],
        [1.2819],
        [0.2303]])
encoder_lstm.weight_ih_l0 tensor([[ 2.7584],
        [ 1.2672],
        [-0.3520],
        [ 1.8275]])
encoder_lstm.weight_hh_l0 tensor([[-1.0620],
        [-1.9644],
        [ 1.6370],
        [-1.7457]])
encoder_lstm.bias_ih_l0 tensor([ 2.4830,  1.9250, -1.8226,  1.7405])
encoder_lstm.bias_hh_l0 tensor([ 1.2513,  1.6600, -1.7619,  2.1490])
decoder_we.weight tensor([[ 1.1103],
        [-2.5975],
        [-0.9890],
        [ 2.6594]])
decoder_lstm.weight_ih_l0 tensor([[-1.1019],
        [ 2.4490],
        [-2.0733],
        [-0.2866]])
decoder_lstm.weight_hh_l0 tensor([[-1.2341],
        [ 1.2747],
        [-0.2080],
        [-2.1133]])
decoder_lstm.bias_ih_l0 tensor([ 1.8363, -0.7282,  0.4856,  2.8957])
decoder_lstm.bias_hh_l0 tensor([ 2.1457, -0.8942,  0.6420,  2.1818])
decoder_fc.weight tensor([[ 2.2585,  0.3522],
        [-1.4836, -2.4880],
        [ 1.4663, 

In [8]:
# We save the weights
trainer.save_checkpoint("seq2seq_attention_en2es_52_trained.ckpt")

`weights_only` was not set, defaulting to `False`.
